# Notebook 1 — Yousuf (pipeline)

**What this does:** loads the English FLORES+ sentences, finds all the proper names in them (Stage 1), and builds a "fair scorer" that removes copied names before scoring (Stage 3).

**How to use:** paste your Hugging Face token in Cell 2, then press **Runtime -> Run all**. Read what each cell prints before moving on.

**Before running:** make sure you accepted the dataset terms at https://huggingface.co/datasets/openlanguagedata/flores_plus

## Cell 1 — install the tools (takes ~2 min)

In [1]:
!pip install -q datasets spacy sacrebleu
!python -m spacy download en_core_web_sm -q
print('Tools installed.')

'pip' is not recognized as an internal or external command,
operable program or batch file.


Tools installed.


Python was not found; run without arguments to install from the Microsoft Store, or disable this shortcut from Settings > Apps > Advanced app settings > App execution aliases.


## Cell 2 — log in (CHANGE THE TOKEN LINE)

In [2]:
from huggingface_hub import login
login('PASTE_YOUR_TOKEN_HERE')   # <-- change ONLY this
print('Logged in.')

Logged in.


## Cell 3 — load English sentences (should print ~1012)

In [3]:
from datasets import load_dataset

eng = load_dataset('openlanguagedata/flores_plus', 'eng_Latn', split='devtest')
english = [r['text'] for r in eng]
print('Loaded', len(english), 'English sentences')
print('Example:', english[0])

Resolving data files:   0%|          | 0/227 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/221 [00:00<?, ?it/s]

Loaded 1012 English sentences
Example: "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.


## Cell 4 — STAGE 1: find the names in every sentence
This marks people, places, organisations, numbers, and dates. These are what models 'cheat' by copying.

In [4]:
import spacy
nlp = spacy.load('en_core_web_sm')

# For each sentence, collect the entity text strings.
all_entities = []
for sentence in english:
    doc = nlp(sentence)
    names = [ent.text for ent in doc.ents]
    all_entities.append(names)

# Show a few so you can eyeball accuracy
for i in range(5):
    print('SENTENCE:', english[i])
    print('NAMES   :', all_entities[i])
    print('-' * 60)

total = sum(len(x) for x in all_entities)
print('Total names found across all sentences:', total)

SENTENCE: "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.
NAMES   : ['4-month-old']
------------------------------------------------------------
SENTENCE: Dr. Ehud Ur, professor of medicine at Dalhousie University in Halifax, Nova Scotia and chair of the clinical and scientific division of the Canadian Diabetes Association cautioned that the research is still in its early days.
NAMES   : ['Ehud Ur', 'Dalhousie University', 'Halifax', 'Nova Scotia', 'the Canadian Diabetes Association', 'early days']
------------------------------------------------------------
SENTENCE: Like some other experts, he is skeptical about whether diabetes can be cured, noting that these findings have no relevance to people who already have Type 1 diabetes.
NAMES   : ['1']
------------------------------------------------------------
SENTENCE: On Monday, Sara Danius, permanent secretary of the Nobel Committee for Literature at the Swedish Academy, publicly announced durin

## Cell 5 — save the names to a file (put this in your repo)

In [5]:
import json
with open('english_entities.json', 'w') as f:
    json.dump(all_entities, f)
print('Saved english_entities.json  (download it and add to the GitHub repo)')

Saved english_entities.json  (download it and add to the GitHub repo)


## Cell 6 — STAGE 3: the 'fair scorer'
This removes any name that was copied straight from the English into the translation (and from the answer key), then scores what's left. The gap between the normal score and this fair score is the **inflation** — your paper's main number.

Run this cell to define the function; the next cell tests it.

In [6]:
import sacrebleu

def remove_copied_names(text, names):
    """Delete any English name that appears verbatim in this line of text."""
    for name in names:
        if name and name in text:
            text = text.replace(name, ' ')
    # tidy up double spaces
    return ' '.join(text.split())

def fair_and_normal_scores(translations, answers, entities_per_sentence):
    """Return (normal BLEU, fair BLEU, normal chrF, fair chrF)."""
    # NORMAL: score as-is
    normal_bleu = sacrebleu.corpus_bleu(translations, [answers]).score
    normal_chrf = sacrebleu.corpus_chrf(translations, [answers]).score

    # FAIR: strip copied English names from BOTH sides first
    masked_tr, masked_ans = [], []
    for tr, ans, names in zip(translations, answers, entities_per_sentence):
        masked_tr.append(remove_copied_names(tr, names))
        masked_ans.append(remove_copied_names(ans, names))
    fair_bleu = sacrebleu.corpus_bleu(masked_tr, [masked_ans]).score
    fair_chrf = sacrebleu.corpus_chrf(masked_tr, [masked_ans]).score

    return normal_bleu, fair_bleu, normal_chrf, fair_chrf

print('Fair scorer ready.')

Fair scorer ready.


## Cell 7 — quick test of the fair scorer on 3 scripts
Here the 'translation' is just the copied English (the cheat). Normal score will be higher than fair score — that difference is inflation. We test Spanish (Latin), Bengali, and Arabic.

In [7]:
test_langs = ['spa_Latn', 'ben_Beng', 'arb_Arab']

for lang in test_langs:
    answers = [r['text'] for r in
               load_dataset('openlanguagedata/flores_plus', lang, split='devtest')]
    # the cheat: pretend copied English is the translation
    n_bleu, f_bleu, n_chrf, f_chrf = fair_and_normal_scores(english, answers, all_entities)
    print(lang)
    print('  normal BLEU:', round(n_bleu, 2), ' fair BLEU:', round(f_bleu, 2),
          ' inflation:', round(n_bleu - f_bleu, 2))
    print('  normal chrF:', round(n_chrf, 2), ' fair chrF:', round(f_chrf, 2),
          ' inflation:', round(n_chrf - f_chrf, 2))
    print('-' * 60)

Resolving data files:   0%|          | 0/227 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/221 [00:00<?, ?it/s]

spa_Latn.jsonl:   0%|          | 0.00/449k [00:00<?, ?B/s]

C:\Users\younu\.gemini\antigravity-ide\scratch\nlp_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\younu\.cache\huggingface\hub\datasets--openlanguagedata--flores_plus. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spa_Latn.jsonl:   0%|          | 0.00/468k [00:00<?, ?B/s]

...

Generating devtest split: 0 examples [00:00, ? examples/s]

That's 100 lines that end in a tokenized period ('.')


It looks like you forgot to detokenize your test data, which may hurt your score.


If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


arb_Arab
  normal BLEU: 0.61  fair BLEU: 0.36  inflation: 0.25
  normal chrF: 0.79  fair chrF: 0.38  inflation: 0.41
------------------------------------------------------------


## What to check
- Names in Cell 4 should look like real names/places/numbers.
- In Cell 7, `inflation` should be a positive number — bigger for Spanish (Latin script copies easily) than for Bengali/Arabic if the hypothesis holds.
- Download `english_entities.json` (folder icon on the left -> three dots -> Download) and add it to the repo.

**Next:** once real model translations exist (Stage 4 on Kaggle), you feed them into `fair_and_normal_scores` instead of the copied English, and the inflation numbers become your results.